# LoL-Context Chatbot
### Retrieval-Augmented Generation con Sentence Transformers
**Machine Learning — UNAB 2026 | Prof. Fabio Schwarzemberg**

---

## Arquitectura del sistema

```
Pregunta del usuario
        ↓
  Tokenización + Positional Encoding
        ↓
  Transformer Encoder → Embedding (vector 384d)
        ↓
  Cosine Similarity vs Knowledge Base
  (esto ES Q·Kᵀ / √dk del profe, pero a nivel de documentos)
        ↓
  Respuesta más cercana
```

**Conexión con la clase:** El mecanismo de atención `Attention(Q,K,V) = softmax(QKᵀ/√dk)·V`
opera aquí a escala de documentos:
- **Q** = embedding de la pregunta del usuario
- **K** = embeddings de las preguntas del Knowledge Base  
- **V** = respuestas del Knowledge Base
- **Similaridad** = el score que decide qué respuesta devolver

## 0. Instalación de dependencias
Si usas `uv`: `uv add sentence-transformers numpy matplotlib seaborn`  
Si usas pip directamente:

In [ ]:
# Solo correr si no tienes las dependencias instaladas
# !pip install sentence-transformers matplotlib seaborn numpy

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sentence_transformers import SentenceTransformer

print("Imports OK")

## 2. Modelo de Embeddings

Usamos `paraphrase-multilingual-MiniLM-L12-v2`:
- Soporta español nativo
- Corre local sin GPU (~120MB)
- Basado en arquitectura Transformer (BERT)
- Genera vectores de 384 dimensiones por oración

**Nota:** Este modelo usa internamente el mismo mecanismo de Self-Attention
que vimos en clase. No lo fine-tuneamos — lo usamos como está (frozen).

In [ ]:
print("Cargando modelo... (primera vez descarga ~120MB)")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f"Modelo cargado.")
print(f"Dimensión de embeddings: {model.get_sentence_embedding_dimension()}d")

## 3. Knowledge Base — Corpus LoL

Este es nuestro corpus: 10 pares pregunta/respuesta sobre terminología de League of Legends.
Aquí se aplica el concepto de **polisemia** que explicó el profe:
palabras como `wave` o `canon` tienen un significado completamente distinto fuera del juego.

In [ ]:
knowledge_base = [
    {
        "pregunta": "¿Qué es una wave?",
        "respuesta": "Una wave es la oleada de minions que avanza por el carril cada cierto tiempo. Limpiar la wave significa matar todos los minions enemigos."
    },
    {
        "pregunta": "¿Qué significa backear?",
        "respuesta": "Backear es volver a la base presionando la tecla B. Se hace para recuperar vida, maná o comprar items."
    },
    {
        "pregunta": "¿Qué es el canon?",
        "respuesta": "El canon o minion canonero es el minion grande que aparece cada varias waves. Da más oro que los minions normales y es prioritario matarlo."
    },
    {
        "pregunta": "¿Qué es farmear?",
        "respuesta": "Farmear es matar minions para acumular oro. Un buen farm (CS alto) es clave para comprar items y escalar."
    },
    {
        "pregunta": "¿Qué es un gank?",
        "respuesta": "Un gank es cuando el junglero viene a tu carril a ayudarte a matar al enemigo. Es una emboscada coordinada."
    },
    {
        "pregunta": "¿Qué es el poke?",
        "respuesta": "Pokear es hacerle daño al enemigo desde lejos sin comprometerse en una pelea completa. Se usa para debilitarlo antes de una pelea."
    },
    {
        "pregunta": "¿Qué significa splitpush?",
        "respuesta": "Splitpush es empujar un carril solo mientras tu equipo presiona otro lado del mapa. Es una estrategia de dividir la atención del enemigo."
    },
    {
        "pregunta": "¿Qué es un feeder?",
        "respuesta": "Un feeder es un jugador que muere muchas veces seguidas, dándole oro y ventaja al equipo enemigo. Es algo negativo."
    },
    {
        "pregunta": "¿Qué es el roam?",
        "respuesta": "Roamear es dejar tu carril temporalmente para ir a ayudar a otros carriles. Es común en el rol de support y midlaner."
    },
    {
        "pregunta": "¿Qué es el all-in?",
        "respuesta": "All-in es comprometer todas tus habilidades y recursos en una pelea para intentar matar al enemigo. Es todo o nada."
    }
]

print(f"Knowledge Base cargada: {len(knowledge_base)} entradas")
for i, qa in enumerate(knowledge_base):
    print(f"  [{i+1:02d}] {qa['pregunta']}")

## 4. Generación de Embeddings del KB

Cada pregunta del Knowledge Base pasa por el Transformer y se convierte en un vector de 384 dimensiones.
Este proceso usa internamente:
1. **Tokenización** — texto → tokens numéricos
2. **Positional Encoding** — señal de posición con sin/cos
3. **Self-Attention** — `Attention(Q,K,V) = softmax(QKᵀ/√dk)·V`
4. **Pooling** — el vector final representa toda la oración

In [ ]:
# Extraemos solo las preguntas para encodear
kb_preguntas = [qa["pregunta"] for qa in knowledge_base]
kb_respuestas = [qa["respuesta"] for qa in knowledge_base]

# Generamos los embeddings — aquí el Transformer hace su trabajo
print("Generando embeddings del Knowledge Base...")
kb_embeddings = model.encode(kb_preguntas, normalize_embeddings=True)

print(f"Shape de la matriz de embeddings: {kb_embeddings.shape}")
print(f"  → {kb_embeddings.shape[0]} preguntas × {kb_embeddings.shape[1]} dimensiones")

## 5. Función de Retrieval — Cosine Similarity

La similitud del coseno mide qué tan parecidos son dos vectores:

$$\text{similarity}(Q, K) = \frac{Q \cdot K}{|Q| \cdot |K|}$$

Esto es equivalente al producto punto `Q·Kᵀ` del mecanismo de atención cuando los vectores están normalizados (que es exactamente lo que hace `normalize_embeddings=True` arriba).

**Score = 1.0** → pregunta idéntica  
**Score > 0.7** → pregunta muy similar  
**Score < 0.3** → pregunta no relacionada

In [ ]:
def cosine_similarity(a, b):
    """
    Similitud coseno entre dos vectores.
    Como los embeddings ya están normalizados, esto es solo el producto punto.
    Equivalente a Q·Kᵀ en el mecanismo de atención.
    """
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def retrieve(pregunta_usuario, top_k=1, threshold=0.3):
    """
    Dado un input del usuario:
    1. Lo encodea con el mismo Transformer
    2. Calcula similitud contra todos los embeddings del KB
    3. Devuelve la respuesta más cercana
    """
    # Paso 1: encodear la pregunta del usuario
    query_embedding = model.encode([pregunta_usuario], normalize_embeddings=True)[0]
    
    # Paso 2: calcular similitud contra el KB
    scores = np.array([cosine_similarity(query_embedding, kb_emb) for kb_emb in kb_embeddings])
    
    # Paso 3: encontrar el más cercano
    best_idx = np.argmax(scores)
    best_score = scores[best_idx]
    
    if best_score < threshold:
        return {
            "respuesta": "No entendí esa. Pregunta algo sobre LoL bro.",
            "pregunta_kb": None,
            "score": best_score,
            "scores": scores
        }
    
    return {
        "respuesta": kb_respuestas[best_idx],
        "pregunta_kb": kb_preguntas[best_idx],
        "score": best_score,
        "scores": scores
    }

print("Función de retrieval lista.")

## 6. Demo del Chatbot

Probamos con preguntas en lenguaje coloquial gamer — no necesitan ser iguales a las del KB.

In [ ]:
preguntas_test = [
    "bro qué onda con el canon",
    "cuándo debo backear",
    "qué es una wave de minions",
    "cómo farmeo más",
    "el junglero me puede ayudar?",
    "qué significa que alguien está fedeando",
    "cómo se hace splitpush",
    "qué es roamear",
    "cuándo conviene hacer all in",
    "cómo pokeo sin arriesgarme",
]

print("=" * 60)
print("LOL-CONTEXT CHATBOT")
print("=" * 60)

for pregunta in preguntas_test:
    result = retrieve(pregunta)
    print(f"\nUSER: {pregunta}")
    print(f"  → KB match: {result['pregunta_kb']} (score: {result['score']:.3f})")
    print(f"BOT:  {result['respuesta']}")
    print("-" * 60)

## 7. Visualización 1 — Heatmap de Similitud del KB

Igual que el profe mostró los heatmaps de atención del traductor,
nosotros mostramos la **matriz de similitud** entre todas las preguntas del KB.

**Lo que esperamos ver:**
- Diagonal = 1.0 (cada pregunta es idéntica a sí misma)
- Fuera de diagonal = valores bajos (preguntas distintas no se confunden)
- Si dos celdas off-diagonal son altas → el modelo confunde esas dos preguntas

In [ ]:
# Calcular matriz de similitud completa
n = len(kb_preguntas)
sim_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = cosine_similarity(kb_embeddings[i], kb_embeddings[j])

# Labels cortos para el eje
labels = [
    "wave", "backear", "canon", "farmear", "gank",
    "poke", "splitpush", "feeder", "roam", "all-in"
]

fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    sim_matrix,
    xticklabels=labels,
    yticklabels=labels,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    vmin=0, vmax=1,
    linewidths=0.5,
    ax=ax
)

ax.set_title("Matriz de Similitud — Knowledge Base LoL\n(diagonal = 1.0 es lo esperado)", 
             fontsize=13, pad=15)
ax.set_xlabel("Pregunta KB", fontsize=11)
ax.set_ylabel("Pregunta KB", fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()
print("Si la diagonal es 1.0 y el resto bajo → el modelo distingue bien los conceptos.")

## 8. Visualización 2 — Heatmap de Retrieval por Pregunta

Mostramos qué tan parecida es cada pregunta del usuario a cada entrada del KB.
Esto replica exactamente los heatmaps del traductor del profe:
- Eje Y: preguntas del usuario (input)
- Eje X: entradas del KB
- Color brillante = alta similitud = eso es lo que el bot devuelve

In [ ]:
# Encodear todas las preguntas de test
query_embeddings = model.encode(preguntas_test, normalize_embeddings=True)

# Calcular scores de cada query contra todo el KB
retrieval_matrix = np.zeros((len(preguntas_test), len(kb_preguntas)))

for i, q_emb in enumerate(query_embeddings):
    for j, kb_emb in enumerate(kb_embeddings):
        retrieval_matrix[i][j] = cosine_similarity(q_emb, kb_emb)

# Labels cortos para queries del usuario
query_labels = [
    "qué onda el canon",
    "cuándo backear",
    "qué es wave",
    "cómo farmeo",
    "junglero ayuda?",
    "qué es fedear",
    "cómo splitpush",
    "qué es roamear",
    "cuándo all-in",
    "cómo pokeo",
]

fig, ax = plt.subplots(figsize=(14, 7))

sns.heatmap(
    retrieval_matrix,
    xticklabels=labels,
    yticklabels=query_labels,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0, vmax=1,
    linewidths=0.5,
    ax=ax
)

ax.set_title("Heatmap de Retrieval — Usuario vs Knowledge Base\n(celda brillante = respuesta seleccionada)",
             fontsize=13, pad=15)
ax.set_xlabel("Knowledge Base", fontsize=11)
ax.set_ylabel("Pregunta del usuario", fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()
print("Cada fila debería tener UN punto brillante en la columna correcta.")

## 9. Visualización 3 — Positional Encoding

Replicamos la visualización que mostró el profe.
El Transformer no sabe el orden de las palabras (procesa todo en paralelo),
así que inyecta señales de posición con seno y coseno:

$$PE_{(pos,\ 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos,\ 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
def positional_encoding(max_len, d_model):
    PE = np.zeros((max_len, d_model))
    for pos in range(max_len):
        for i in range(0, d_model, 2):
            PE[pos, i]     = np.sin(pos / (10000 ** (2 * i / d_model)))
            if i + 1 < d_model:
                PE[pos, i+1] = np.cos(pos / (10000 ** (2 * i / d_model)))
    return PE

# Visualizar para una oración de 20 tokens y 64 dimensiones
PE = positional_encoding(max_len=20, d_model=64)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(PE, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_title("Positional Encoding — señal de posición inyectada en cada token\n(cada fila = un token, cada columna = una dimensión)",
             fontsize=12, pad=12)
ax.set_xlabel("Dimensión del embedding", fontsize=11)
ax.set_ylabel("Posición del token en la frase", fontsize=11)
plt.tight_layout()
plt.show()
print("Cada token tiene una firma única de posición. El modelo aprende a distinguirlas.")

## 10. Chatbot Interactivo

Loop de conversación. Escribe `salir` para terminar.

In [ ]:
print("=" * 50)
print("  LOL-CONTEXT BOT  |  'salir' para terminar")
print("=" * 50)

while True:
    user_input = input("\nTú: ").strip()
    
    if user_input.lower() in ["salir", "exit", "quit", ""]:
        print("Bot: gg wp bro")
        break
    
    result = retrieve(user_input)
    print(f"Bot: {result['respuesta']}")
    print(f"     [score: {result['score']:.3f} | match: {result['pregunta_kb']}]")

---
## Resumen de conceptos aplicados

| Concepto (clase) | Dónde se aplica en este proyecto |
|---|---|
| Polisemia | `wave`, `canon`, `backear` tienen significado distinto al lenguaje común |
| Embeddings contextuales | `SentenceTransformer` genera vectores según contexto, no estáticos |
| Positional Encoding | Visualizado en la celda 9, aplicado internamente por el modelo |
| Q, K, V Attention | Q = query del usuario, K = preguntas KB, V = respuestas KB |
| Scaled Dot Product | `cosine_similarity` ≡ `Q·Kᵀ/√dk` con vectores normalizados |
| Heatmap de atención | Visualizaciones 7 y 8: similitud KB y retrieval por query |
| RAG | Retrieval del KB + respuesta = Retrieval-Augmented Generation |